In [1]:
import os

import kagglehub
import pandas as pd
from sklearn.compose import ColumnTransformer


def kaggle_dataset(url):
    dataset_path = kagglehub.competition_download(url)
    train_df = pd.read_csv(os.path.join(dataset_path, "train.csv"))
    test_df = pd.read_csv(os.path.join(dataset_path, "test.csv"))
    return train_df, test_df

In [ ]:
# ensemble method does well at averaging the noise of the models.

In [2]:
kagglehub.competition_download('playground-series-s6e4')

100%|██████████| 32.5M/32.5M [00:03<00:00, 9.44MB/s]

Extracting files...


'/home/zak/.cache/kagglehub/competitions/playground-series-s6e4'

In [5]:
train, test = kaggle_dataset('playground-series-s6e4')

In [66]:
train

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,629995,Clay,6.54,13.45,1.15,1.86,26.65,26.86,1041.33,10.62,...,Rice,Sowing,Kharif,Sprinkler,River,4.35,No,118.36,South,Medium
629996,629996,Clay,7.03,54.49,0.96,2.35,36.99,88.00,1419.57,9.93,...,Sugarcane,Vegetative,Kharif,Drip,Groundwater,12.97,Yes,40.75,Central,Medium
629997,629997,Clay,6.52,11.98,0.93,0.38,37.82,70.98,88.45,8.19,...,Potato,Vegetative,Zaid,Canal,Reservoir,13.58,Yes,2.62,South,High
629998,629998,Clay,5.93,42.86,0.33,3.39,34.99,94.58,2433.92,9.88,...,Sugarcane,Vegetative,Zaid,Sprinkler,Groundwater,0.79,Yes,90.00,East,Low


In [6]:
train.Irrigation_Need.value_counts()
# 3

Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64

In [7]:
from sklearn.pipeline import Pipeline

## Let's Do Some EDA

1. The Data
    - Missing Values
    - Data Types
    - Errors
2. Handling Missing Data
    - Why is the data missing? Missingness.
    - Remove or Impute?
    - Suitable Imputation Method
    - Make a note of any bias the missing data may introduce.
3. Data Distribution
    - Measure of central tendancy
    - Measure virability and standard deviation
    - Skewness and kurtosis
    - Outliers and Anomalies
4. Data Transformations
    - Scaling or Normalization
    - Encoding categorical data (one-hot or label encoding)
    - Mathematical transformations
    - Creating new features
    - Aggregating data
5. Data Relationships
    - Categorical variables

In [8]:
train.select_dtypes(str).nunique()
# float64 and str. The id column is in64 but that should be removed/contain no predictive information.


Soil_Type            4
Crop_Type            6
Crop_Growth_Stage    4
Season               3
Irrigation_Type      4
Water_Source         4
Mulching_Used        2
Region               5
Irrigation_Need      3
dtype: int64

In [48]:
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder

## Handling Categorical Data
On first inspection the target variable consists of three values, Low, Medium and High. This feels like ordinal data and could be transformed in the values 1, 2 and 3.


In [49]:
categorical_pipeline = Pipeline(steps=[
    ('encoder', OneHotEncoder())
])

## Handling Numeric Data

Create a pipeline to handle numeric data. In this case we'll be scaling.

In [55]:
numeric_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

In [65]:
# Set benchmark to some linear model etc. What features could be more powerful to the model?

In [56]:
categorical_cols = train.select_dtypes(include=[str]).columns.drop('Irrigation_Need')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, train.select_dtypes(include=[float]).columns),
        ('cat', categorical_pipeline, categorical_cols),
    ])

In [57]:
pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier())
    ])

In [58]:
y = train.Irrigation_Need
X = train.drop('Irrigation_Need', axis=1)

In [59]:
results = cross_validate(pipeline, X, y, cv=5, scoring='accuracy')

In [60]:
results

{'fit_time': array([7.29743218, 7.25007486, 7.32148552, 7.27239013, 7.29653478]),
 'score_time': array([0.16476989, 0.17080092, 0.17147326, 0.17423511, 0.17127848]),
 'test_score': array([0.97080159, 0.97078571, 0.97096032, 0.96997619, 0.97020635])}

In [61]:
class Project:
    """
    A way of managing the project and keeping track of pipelines.
    """

    def __init__(self, project_name: str, dataframe: pd.DataFrame, target_column: str, split_data: bool = True,
                 standardize_columns=True):
        # TODO: Maybe and some standard column naming to the columns
        self.name = project_name
        self._raw_data = dataframe.copy()
        self.target_column = target_column
        self.X = self._raw_data.drop(self.target_column, axis=1)
        self.y = self._raw_data[target_column]

        if split_data:
            # TODO: Implement train test split. Not needed in first version when using kaggle comp data
            pass
        else:
            self.training_data = self._raw_data

    def get_training_data(self):
        return self.training_data

    def eda(self):
        # TODO: Complete some eda at the project level. These things should be high enough level that you would want the recommendations applied to all the experiments
        """

        :return:
        """

        return


In [62]:
irrigation_needs = Project('irrigation_needs', train, target_column='Irrigation_Need', split_data=False, standardize_columns=False)

In [63]:
irrigation_needs.X.head()
irrigation_needs.y.head()

0       Low
1       Low
2       Low
3    Medium
4       Low
Name: Irrigation_Need, dtype: str

In [64]:
# import time
# import math
#
# t1 = time.time()
#
# # Normal
# r = [math.factorial(int(math.sqrt(i ** 3))) for i in range(100, 1000)]
#
# t2 = time.time()
#
# print(t2 - t1)

3.391441583633423


In [15]:
import time
from joblib import Parallel, delayed
import math

t1 = time.time()

# 2 Core
r1 = Parallel(n_jobs=-1)(delayed(math.factorial)(int(math.sqrt(i ** 3))) for i in range(100, 1000))

t2 = time.time()

print(t2 - t1)

0.6652145385742188


In [16]:
from joblib import Memory

location = 'cache_dir'

memory = Memory(location=location, verbose=0)


@memory.cache
def expensive_function(x=1000):
    r = [math.factorial(int(math.sqrt(i ** 3))) for i in range(100, x)]
    return r


result1 = expensive_function()

In [17]:
result2 = expensive_function()

In [18]:
from shutil import rmtree

# Delete the temporary cache before exiting
memory.clear()
rmtree(location)

[Memory(location=cache_dir/joblib)]: Flushing completely the cache


In [19]:
from sklearn.model_selection import cross_validate

In [34]:
class Experiment():
    def __init__(self, name: str, project: Project, model=None):
        self.name = name
        self.X = project.X
        self.y = project.y
        self.model = model

    def run(self):
        """
        Run the experiment, at this point you are accepting the parameters and pipeline. Run will gather the data required to produce the report. Perhaps return something lightweight?
        :return:
        """

        return cross_validate(estimator=self.model, cv=5, X=self.X, y=self.y)

    @property
    def report(self):
        report = 'SOME REPORT ON THE MODEL'
        return report

In [35]:
# TODO: NOTE TO SELF TRY THE DIFFERENT ENCODING OF IRRIGATION_NEED AND SEE WHAT HAPPENS!!!! TRY TO RUN DIFFERENT EXPERIMENTS

In [37]:
experiment_1 = Experiment('experiment_1', irrigation_needs, model=pipeline)

In [38]:
experiment_1.run()

TypeError: If no scoring is specified, the estimator passed should have a 'score' method. The estimator Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', Pipeline(steps=[]),
                                                  Index(['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
       'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours',
       'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm'],
      dtype='str')),
                                                 ('cat', Pipeline(steps=[]),
                                                  Index(['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
       'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region',
       'Irrigation_Need'],
      dtype='str'))])),
                'classifier', DecisionTreeClassifier()]) does not.